# MLP Hyperparameter Optimization

Optuna searches MLP depth, width, activation, and learning rate for the infinite-domain inverse problem. Each completed trial is saved in a timestamped results directory with its models, metrics, and training history.

In [1]:
import os
import sys
from datetime import datetime

from importlib import reload
current_dir = os.getcwd()
utilities_dir = os.path.join(current_dir, '../../utils')
os.chdir(current_dir)
if utilities_dir not in sys.path:
    sys.path.insert(0, utilities_dir)
import plotting
import pinns_infinite
import infinite
reload(plotting)
reload(pinns_infinite)
reload(infinite)
import numpy as np
import sympy as sp
from calflops import calculate_flops
import pandas as pd
import joblib
import matplotlib.pyplot as plt 
import torch
import torch.nn as nn
import torch.optim as optim
from pinns_infinite import  MLP, init_weights, CoefficientNet, pde_loss_inf, observation_loss_u, observation_loss_k, train_dual_network, build_models, set_seed,run_experiment_inf,build_models_KAN
from pinns_infinite import build_models
from infinite import analytical_solution_inf, coefficient_inf, source_term_inf, generate_dataset_inf, evaluate_model_inf
torch.set_default_dtype(torch.float32)
from plotting import plot_histories_comparison

set_seed(1)
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

/home/orincon/miniconda3/envs/PIKAN-unbounded-domains-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
import optuna

# Search space from the architecture-matching study.
MLP_SEARCH_SPACE = {
    "hidden_layers": [1, 2, 3],
    "hidden_units": [15, 90, 104],
    "activation": ["Sine", "Sigmoid", "Tanh"],
    "learning_rate": [1e-4, 1e-3, 1e-2],
}

ACTIVATIONS = {
    "Sine": lambda: Sine(),
    "Sigmoid": nn.Sigmoid,
    "Tanh": nn.Tanh,
}


class Sine(nn.Module):
    def forward(self, x):
        return torch.sin(x)


N_TRIALS = 50
ADAM_ITERS = 2000
LBFGS_ITERS = 2000

# One directory contains the CSV summary and all saved trial models.
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
results_dir = f"results_mlp_infinite_optuna_{timestamp}"
os.makedirs(results_dir, exist_ok=True)

print(f"Results will be saved to: {results_dir}")
print(f"Optuna trials: {N_TRIALS}")

Results will be saved to: results_mlp_optuna_2026-09-20_08-52-35
Optuna trials: 50


## Objective Function

Each trial trains one MLP configuration and minimizes the mean of the global relative errors for `u` and `k`.

In [3]:
def objective(trial):
    """Run one MLP training configuration and return mean global error."""
    config = {
        "hidden_layers": trial.suggest_categorical(
            "hidden_layers",
            MLP_SEARCH_SPACE["hidden_layers"],
        ),
        "hidden_units": trial.suggest_categorical(
            "hidden_units",
            MLP_SEARCH_SPACE["hidden_units"],
        ),
        "activation": trial.suggest_categorical(
            "activation",
            MLP_SEARCH_SPACE["activation"],
        ),
        "learning_rate": trial.suggest_categorical(
            "learning_rate",
            MLP_SEARCH_SPACE["learning_rate"],
        ),
    }
    activation = ACTIVATIONS[config["activation"]]()

    print(
        f"\n--- Trial {trial.number}: "
        f"L={config['hidden_layers']}, "
        f"N={config['hidden_units']}, "
        f"activation={config['activation']}, "
        f"lr={config['learning_rate']:.0e} ---"
    )

    try:
        err_u, err_k, compute_time = run_experiment_inf(
            model_type="MLP",
            hidden_layers=config["hidden_layers"],
            hidden_units=config["hidden_units"],
            activation=activation,
            adam_lr=config["learning_rate"],
            device=device,
            adam_iters=ADAM_ITERS,
            lbfgs_iters=LBFGS_ITERS,
            results_dir=results_dir,
            eval_domain=(-10.0, 10.0, -10.0, 10.0),
        )
    except Exception as error:
        print(f"Trial {trial.number} failed: {error}")
        raise optuna.exceptions.TrialPruned()

    mean_global_error = 0.5 * (err_u + err_k)
    trial.set_user_attr("err_u", float(err_u))
    trial.set_user_attr("err_k", float(err_k))
    trial.set_user_attr("compute_time_sec", float(compute_time))

    print(
        f"Success! Time: {compute_time:.2f}s | "
        f"Err U: {err_u:.3e} | Err K: {err_k:.3e} | "
        f"Mean error: {mean_global_error:.3e}"
    )
    return mean_global_error

## Run Optimization

In [4]:
sampler = optuna.samplers.TPESampler(seed=1)
study = optuna.create_study(
    direction="minimize",
    sampler=sampler,
    study_name=f"mlp_infinite_domain_{timestamp}",
)

study.optimize(
    objective,
    n_trials=N_TRIALS,
    catch=(RuntimeError, ValueError),
)

print("\n========================================")
print("BEST MLP CONFIGURATION")
print("========================================")
print(f"Mean global error: {study.best_value:.6e}")
print("Parameters:")
for name, value in study.best_params.items():
    print(f"  {name}: {value}")

[I 2026-09-20 08:52:38,886] A new study created in memory with name: mlp_infinite_domain_2026-09-20_08-52-35



--- Trial 0: L=2, N=15, activation=Tanh, lr=1e-02 ---


/home/orincon/miniconda3/envs/PIKAN-unbounded-domains-env/lib/python3.12/site-packages/torch/autograd/graph.py:869: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:335.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
[I 2026-09-20 08:53:53,131] Trial 0 finished with value: 0.05457699624546126 and parameters: {'hidden_layers': 2, 'hidden_units': 15, 'activation': 'Tanh', 'learning_rate': 0.01}. Best is trial 0 with value: 0.05457699624546126.



[MLP] L=2, N=15 | Params: 1,082 | Mean Err: 5.458e-02 | Saved to 'results_mlp_optuna_2026-09-20_08-52-35/'.
Success! Time: 74.23s | Err U: 1.764e-02 | Err K: 9.151e-02 | Mean error: 5.458e-02

--- Trial 1: L=2, N=15, activation=Tanh, lr=1e-04 ---


[I 2026-09-20 08:55:18,232] Trial 1 finished with value: 0.06820994108433058 and parameters: {'hidden_layers': 2, 'hidden_units': 15, 'activation': 'Tanh', 'learning_rate': 0.0001}. Best is trial 0 with value: 0.05457699624546126.


...


[MLP] L=2, N=15 | Params: 1,082 | Mean Err: 7.175e-02 | Saved to 'results_mlp_optuna_2026-09-20_08-52-35/'.
Success! Time: 84.52s | Err U: 3.484e-02 | Err K: 1.087e-01 | Mean error: 7.175e-02

--- Trial 48: L=1, N=90, activation=Tanh, lr=1e-03 ---


[I 2026-09-20 10:03:42,055] Trial 48 finished with value: 0.01294751804110555 and parameters: {'hidden_layers': 1, 'hidden_units': 90, 'activation': 'Tanh', 'learning_rate': 0.001}. Best is trial 19 with value: 0.0013199545212951597.



[MLP] L=1, N=90 | Params: 17,102 | Mean Err: 1.295e-02 | Saved to 'results_mlp_optuna_2026-09-20_08-52-35/'.
Success! Time: 78.28s | Err U: 2.179e-02 | Err K: 4.105e-03 | Mean error: 1.295e-02

--- Trial 49: L=3, N=104, activation=Sine, lr=1e-02 ---


[I 2026-09-20 10:04:07,312] Trial 49 finished with value: 0.7094200984193925 and parameters: {'hidden_layers': 3, 'hidden_units': 104, 'activation': 'Sine', 'learning_rate': 0.01}. Best is trial 19 with value: 0.0013199545212951597.



[MLP] L=3, N=104 | Params: 66,354 | Mean Err: 7.094e-01 | Saved to 'results_mlp_optuna_2026-09-20_08-52-35/'.
Success! Time: 25.25s | Err U: 2.165e-02 | Err K: 1.397e+00 | Mean error: 7.094e-01

BEST MLP CONFIGURATION
Mean global error: 1.319955e-03
Parameters:
  hidden_layers: 3
  hidden_units: 90
  activation: Sine
  learning_rate: 0.001


## Save Optimization Results

In [5]:
# Persist the complete study and a tabular summary for later analysis.
data_dir = os.path.join(results_dir, "data")
os.makedirs(data_dir, exist_ok=True)

joblib.dump(study, os.path.join(data_dir, "study.pkl"))
joblib.dump(study, os.path.join(data_dir, f"study_{timestamp}.pkl"))

study_df = study.trials_dataframe()
study_csv_path = os.path.join(data_dir, "study.csv")
study_df.to_csv(study_csv_path, index=False)

completed_df = study_df[
    study_df["state"].eq("COMPLETE")
].sort_values(by="value", ascending=True)
completed_csv_path = os.path.join(data_dir, "study_completed_sorted.csv")
completed_df.to_csv(completed_csv_path, index=False)

print(f"Saved study to: {data_dir}")
print(f"Saved trial summary to: {study_csv_path}")
print(f"Saved sorted completed trials to: {completed_csv_path}")

Saved study to: results_mlp_optuna_2026-09-20_08-52-35/data
Saved trial summary to: results_mlp_optuna_2026-09-20_08-52-35/data/study.csv
Saved sorted completed trials to: results_mlp_optuna_2026-09-20_08-52-35/data/study_completed_sorted.csv
